<a href="https://colab.research.google.com/github/writetomangamsudheer-stack/ai-mentor-portfolio/blob/main/Day2_ResumeExtractor_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [3]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [4]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'Extract a Resume JSON from this text. Return ONLY JSON, no markdown.\n\n{raw_text}',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == max_retries:
                raise
            # Retry once with the broken JSON in the prompt
            fix_prompt = (f'Fix this JSON to match schema. Errors: {e}. '
                          f'Original: {resp.text}')
            resp = client.models.generate_content(
                model='gemini-2.5-flash', contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )
            return Resume.model_validate_json(resp.text)

In [5]:
with open('/content/sample_data/sample_resumes.txt') as f:
    resumes = [r.strip() for r in f.read().split('---') if r.strip()]

print(f'Loaded {len(resumes)} sample résumés')

results = []
for i, r in enumerate(resumes[:3]):
    try:
        parsed = extract_resume(r)
        results.append(parsed)
        print(f'\nRésumé {i+1}: {parsed.name} — {len(parsed.skills)} skills, '
              f'{parsed.experience_years} years exp')
    except Exception as e:
        print(f'\nRésumé {i+1}: FAILED — {type(e).__name__}: {str(e)[:200]}')

# Print full first result
if results:
    print('\n=== Full first result ===')
    print(results[0].model_dump_json(indent=2))

Loaded 5 sample résumés

Résumé 1: Ravi Kumar — 6 skills, 0.25 years exp

Résumé 2: Sneha Reddy — 6 skills, 0.12 years exp

Résumé 3: Arun Pillai — 8 skills, 0.5 years exp

=== Full first result ===
{
  "name": "Ravi Kumar",
  "email": "ravi.kumar@example.com",
  "phone": "+91-9876543210",
  "education": [
    {
      "degree": "B.Tech Computer Science",
      "institution": "Aditya University",
      "year": 2026
    },
    {
      "degree": "Intermediate",
      "institution": "Narayana Junior College",
      "year": 2022
    }
  ],
  "skills": [
    "Python",
    "Java",
    "SQL",
    "Git",
    "Linux",
    "REST APIs"
  ],
  "projects": [
    "Built a Flask REST API for college placement portal (3-week internship at startup)",
    "Solved 250+ DSA problems on LeetCode (Top 5% in college)",
    "Final-year project: ML model for crop yield prediction (Random Forest, 87% accuracy)"
  ],
  "experience_years": 0.25
}


In [10]:
from google import genai
from pydantic import ValidationError

# Create Gemini client
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

# Read MCQ text file
with open('/content/sample_data/10_mcq_questions_with_answers.txt') as f:
    raw_text = f.read()

try:
    # Send to Gemini
    response = client.models.generate_content(
        model='gemini-2.5-flash',

        contents=f"""
Extract all MCQs from this text.

Return ONLY valid JSON.

MCQ text:
{raw_text}
""",

        config={
            'response_mime_type': 'application/json',
            'response_schema': MCQList.model_json_schema(),
            'temperature': 0
        }
    )

    # Validate JSON using Pydantic
    parsed = MCQList.model_validate_json(response.text)

    print("Extraction successful!\n")

    # Print formatted JSON
    print(parsed.model_dump_json(indent=2))

    # Save JSON file
    with open('/content/mcqs.json', 'w') as f:
        f.write(parsed.model_dump_json(indent=2))

    print("\nJSON file saved as mcqs.json")

except ValidationError as e:
    print("Validation Error:")
    print(e)

except Exception as e:
    print("Other Error:")
    print(e)

Extraction successful!

{
  "mcqs": [
    {
      "question": "Which data structure follows the LIFO principle?",
      "options": {
        "A": "Queue",
        "B": "Stack",
        "C": "Array",
        "D": "Linked List"
      },
      "answer": "B"
    },
    {
      "question": "Which keyword is used to create a class in Java?",
      "options": {
        "A": "object",
        "B": "struct",
        "C": "class",
        "D": "define"
      },
      "answer": "C"
    },
    {
      "question": "What is the time complexity of binary search in a sorted array?",
      "options": {
        "A": "O(n)",
        "B": "O(log n)",
        "C": "O(n log n)",
        "D": "O(1)"
      },
      "answer": "B"
    },
    {
      "question": "Which of the following is a primary key property in DBMS?",
      "options": {
        "A": "It can contain duplicate values",
        "B": "It can contain NULL values",
        "C": "It uniquely identifies each record",
        "D": "It is used only fo

In [9]:
from pydantic import BaseModel
from typing import List

class Options(BaseModel):
    A: str
    B: str
    C: str
    D: str

class MCQ(BaseModel):
    question: str
    options: Options
    answer: str

class MCQList(BaseModel):
    mcqs: List[MCQ]